# Spam Classifier (Logistic Regression from Scratch)

Binary classification model for detecting spam messages using:
- Logistic regression
- Cross-entropy loss
- Gradient descent
- Evaluation metrics (precision, recall)

Designed to be modular and later converted into a Python service for API deployment.

In [78]:
#useful inputs
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import re

In [79]:
#load the dataset
df = pd.read_csv("dataset/spam.csv", encoding='latin-1')[['v1','v2']]
df.columns = ['label', 'message']
df['label'] = df['label'].str.strip().str.lower()
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

spam = df[df['label'] == 1]
ham = df[df['label'] == 0]

ham_sampled = ham.sample(len(spam), random_state=42)

df = pd.concat([spam, ham_sampled])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [80]:
print(df.shape)
print(df['label'].value_counts())

(1494, 2)
label
0    747
1    747
Name: count, dtype: int64


In [81]:
#data cleaing i.e. texts
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]','', text)
    return text

df['cleaned_msg'] = df['message'].apply(clean_text)


#limited vocab
all_words= []
for text in df['cleaned_msg']:
    all_words.extend(text.split())

common_words = Counter(all_words).most_common(1000)
vocab = {word: i for i, (word, _) in enumerate(common_words)}

In [82]:
#vectorization and normalization
def vectorize(text):
    vec = np.zeros(len(vocab))
    for word in text.split():
        if word in vocab:
            vec[vocab[word]] += 1
    return vec

X = np.array([vectorize(text) for text in df['cleaned_msg']])
Y = df['label'].values




In [83]:
#training and testing split
idx = np.random.permutation(len(X))

X = X[idx]
Y = Y[idx]

split = int(0.8 * len(X))
split = int(0.7 * len(X))

x_train, x_test = X[:split], X[split:]
y_train, y_test = Y[:split], Y[split:]

## Logistic Regression Model

- Sigmoid for probability estimation
- Cross-entropy loss for optimization
- Gradient descent for parameter updates

In [84]:
#sigmoid and loss functions
def sigmoid(z):
    """Compute the sigmoid function"""
    return 1 / (1 + np.exp(-z))


def cross_entropy_loss(y_true, p_pred):
    eps = 1e-15
    p_pred = np.clip(p_pred, eps, 1 - eps)
    return -np.mean(y_true*np.log(p_pred) + (1-y_true)*np.log(1-p_pred))

In [85]:
#training 
w = np.zeros(x_train.shape[1])
b = 0

lr = 0.1
epochs = 350

for i in range(epochs):
    Z = x_train @ w + b
    p = sigmoid(Z)
    
    dw = (1/len(x_train)) * (x_train.T @ (p - y_train))
    db = (1/len(x_train)) * np.sum(p - y_train)
    
    w -= lr * dw
    b -= lr * db

## Evaluation Metrics

Evaluate model performance using:
- Confusion matrix
- Precision
- Recall

In [86]:
#predicitons
Z = x_test @ w + b
p = sigmoid(Z)

threshold = 0.3
y_pred = (p >= threshold).astype(int)

In [87]:
#metrics
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0

print("Precision:", precision)
print("Recall:", recall)

Precision: 0.8664122137404581
Recall: 0.9659574468085106


In [88]:
def predict_spam(text):
    text = clean_text(text)

    vec = np.zeros(len(vocab))
    for word in text.split():
        if word in vocab:
            vec[vocab[word]] += 1
    z = np.dot(vec, w) + b
    p = 1 / (1 + np.exp(-z))

    print("DEBUG → z:", z, "p:", p)  

    return "Most Likely Spam" if p >= 0.4 else "Most Likely Ham"

print(predict_spam("Congratulations! You won a free prize"))
print(predict_spam("Let's meet tomorrow at 10"))

DEBUG → z: 0.2220959576507857 p: 0.5552968754220403
Most Likely Spam
DEBUG → z: -1.429990272216297 p: 0.19310019993911334
Most Likely Ham
